# Netflix Customer Churn: Business Understanding

## From customer behavior to retention action

This notebook defines the business problem, the decisions the analysis should support, and the role of predictive modeling in a customer retention workflow. It is the starting point for the exploratory analysis, churn model, and Streamlit application in this project.

> **Project outcome:** estimate which customer profiles are more likely to churn so a retention team can prioritize attention and design more relevant interventions.

## 1. Business Context

Netflix operates a subscription-based service where long-term value depends on customers continuing their memberships. When a customer cancels or becomes inactive, the business may lose recurring revenue and also lose the opportunity to build a longer-term relationship with that customer.

Churn is rarely caused by one isolated event. It can be associated with reduced watch time, fewer sessions, low interaction with recommendations, declining ratings, a long period since the last login, or a mismatch between the customer and their subscription experience. These signals are spread across customer, account, content, device, and engagement data.

A useful retention process therefore needs more than a list of customers who already left. It needs an earlier risk signal that helps the business decide where proactive engagement may be worthwhile.

## 2. Problem Statement

### Business problem

The retention team does not have a consistent, data-informed way to identify customers who may be at risk of churn and prioritize limited outreach resources. A broad campaign can be expensive and poorly targeted, while waiting until cancellation is too late.

### Analytical problem

Given a customer profile containing demographic, subscription, account, content, engagement, and recency attributes, estimate the probability that the customer belongs to the churned class.

### Decision problem

For each customer or customer profile, the business needs a practical signal that can support questions such as:

- Should this customer be considered for a retention action?
- Which customer groups show similar risk or engagement patterns?
- Which behaviors should be investigated before designing an intervention?
- Is the expected value of contacting this customer greater than the cost of the action?

## 3. Business Objectives

The project is designed to support five connected objectives:

1. **Detect risk earlier:** identify customers whose behavior resembles previously churned customers.
2. **Prioritize retention effort:** rank or classify profiles so teams can focus on higher-risk customers first.
3. **Understand customer behavior:** use exploratory analysis to connect engagement and recency patterns with churn.
4. **Enable practical testing:** provide an interactive application that allows a stakeholder to enter a profile and inspect the predicted outcome.
5. **Create a foundation for segmentation:** group customers by behavioral similarity in a future unsupervised-learning workflow so retention actions can be more relevant to each group.

## 4. Stakeholders and Decisions

| Stakeholder | Business question | How this project helps |
| --- | --- | --- |
| Retention or CRM team | Which customers should be contacted first? | Provides a churn-risk prediction and probability. |
| Product team | Which usage patterns are associated with disengagement? | Surfaces relationships among activity, content, device, and churn. |
| Content and recommendation teams | Are customers engaging with discovery and viewing experiences? | Includes genre, recommendation source, click rate, completion, and watch-time signals. |
| Customer experience team | Which customer profiles may need support? | Combines account, subscription, rating, and recency information. |
| Data and analytics team | Can the workflow be reproduced and deployed? | Uses a documented pipeline, saved model artifact, and Streamlit interface. |
| Business leadership | Is retention investment producing value? | Creates a foundation for measuring targeted interventions against churn outcomes. |

## 5. Project Scope

### In scope

- Analyze a prepared dataset of 50,000 customer records.
- Understand the target variable `churned`, where `1` means churned and `0` means not churned.
- Explore demographic, account, subscription, device, content, engagement, ratings, and recency variables.
- Build and compare classification models.
- Save a complete preprocessing-and-model pipeline.
- Provide an interactive single-profile prediction experience through Streamlit.

### Out of scope for the current version

- Automatic customer outreach or campaign execution.
- Causal claims that a specific behavior causes churn.
- Production-scale batch scoring, customer history, or CRM integration.
- A completed clustering-based segmentation application.
- Individual-level explanations such as SHAP values.

The current product is a decision-support prototype. It provides a risk signal that should be combined with business rules, customer context, and measured retention experiments.

## 6. Available Data and Business Meaning

The dataset contains 20 columns: one customer identifier, 18 candidate predictors, and the binary churn target. The predictor groups are:

| Business area | Variables | Why they matter |
| --- | --- | --- |
| Customer profile | `age`, `gender`, `region` | Helps compare risk across broad customer profiles. |
| Subscription and account | `subscription_type`, `payment_method`, `account_age_months` | Represents plan context, payment relationship, and tenure. |
| Device and content | `primary_device`, `favorite_genre`, `time_of_day` | Describes how and when customers consume the service. |
| Discovery experience | `recommendation_source`, `recommendation_click_rate` | Indicates whether recommendations are being noticed and used. |
| Engagement | `session_count`, `avg_watch_time_minutes_per_week`, `watch_sessions_per_week`, `completion_rate` | Captures depth, frequency, and quality of usage. |
| Feedback and satisfaction signals | `avg_rating_given`, `app_rating` | Provides behavioral and stated feedback signals. |
| Recency | `days_since_last_login` | Indicates how recently the customer interacted with the service. |
| Outcome | `churned` | Historical label used to train and evaluate the classifier. |

`user_id` is an identifier rather than a behavioral signal, so it is excluded from model training.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path.cwd().parent / "Data" / "netflix_user_behavior_churn_50000v2.csv"
df = pd.read_csv(data_path)

profile = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Churned customers", "Not churned customers", "Churn rate (%)", "Missing values", "Duplicate rows"],
    "Value": [
        len(df),
        df.shape[1],
        int((df["churned"] == 1).sum()),
        int((df["churned"] == 0).sum()),
        round(df["churned"].mean() * 100, 2),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
})
profile

,Metric,Value
0,Rows,50000.00
1,Columns,20.00
2,Churned customers,10464.00
3,Not churned customers,39536.00
4,Churn rate (%),20.93
5,Missing values,0.00
6,Duplicate rows,0.00


## 7. Initial Business Understanding

The current dataset provides a strong starting point for a classification problem:

- There are 50,000 records and 20 columns.
- The churn rate is approximately 20.93%, so the classes are not evenly balanced. Accuracy alone should not determine model quality.
- There are no missing values or duplicate rows identified during the initial data review.
- The data combines customer attributes with actionable engagement and recency signals.
- The presence of `days_since_last_login`, watch activity, completion, and recommendation interaction makes the dataset suitable for an early-warning use case.
- Because the data is prepared or synthetic, findings should be treated as a project demonstration and analytical starting point, not as verified production behavior.

## 8. Proposed Analytical Solution

The solution follows a progression from understanding to action:

```text
Business question
      |
      v
Customer behavior data
      |
      v
Data quality and exploratory analysis
      |
      v
Preprocessing + churn classification
      |
      v
Churn probability for a customer profile
      |
      v
Retention prioritization and targeted experimentation
```

The modeling workflow compares a Logistic Regression baseline, Random Forest, and XGBoost. Numerical features are standardized, categorical features are one-hot encoded, and the preprocessing is stored with the final model. This makes the Streamlit prediction path consistent with the training path.

The application returns both a class prediction and a probability. The probability is particularly useful for prioritization because a retention team can later choose an operating threshold based on campaign capacity and the relative cost of missed churners versus unnecessary outreach.

## 9. Success Measures

### Model success

Model evaluation should include:

- **Recall:** how many actual churners are identified, important when missing a high-risk customer is expensive.
- **Precision:** how many flagged customers are genuinely churn-prone, important when outreach capacity is limited.
- **F1 score:** a balance between precision and recall.
- **ROC-AUC and PR-AUC:** ranking quality, with PR-AUC especially useful for an imbalanced target.
- **Calibration:** whether predicted probabilities correspond to observed churn rates.

### Business success

A production version should ultimately be measured by business outcomes rather than model metrics alone:

- Lower churn among customers who receive a targeted intervention
- Incremental retention or revenue compared with a control group
- Cost per retained customer
- Contact efficiency and campaign capacity utilization
- Stability of performance across customer segments and over time

The correct operational threshold should be selected using these costs and outcomes, not chosen arbitrarily.

## 10. From Prediction to Retention Action

A prediction is useful only when it supports a responsible action. A possible operating workflow is:

1. Score eligible customers using the latest available behavioral data.
2. Rank customers by predicted churn probability.
3. Apply business rules such as eligibility, recent contact history, value, and consent.
4. Assign an appropriate intervention, such as content discovery support, product education, service recovery, or a carefully tested offer.
5. Hold out a control group and measure incremental outcomes.
6. Feed observed outcomes back into monitoring and future model updates.

The project currently implements steps 1 and part of step 2 for an individual profile through Streamlit. Steps 3 through 6 describe the business workflow required to turn the prototype into a production retention capability.

## 11. Assumptions, Risks, and Responsible Use

### Assumptions

- The historical `churned` label is defined consistently and represents the business outcome of interest.
- The input variables are available before the retention decision is made.
- Customer behavior patterns in future data will be sufficiently similar to the training data.

### Risks

- A model can learn correlations without identifying the true cause of churn.
- A false negative may allow a valuable customer to leave; a false positive may waste contact budget or annoy a customer.
- Behavioral and demographic variables may create uneven performance across groups.
- Data drift, changes in pricing, content, product design, or customer mix can reduce model reliability.
- The prepared dataset may not represent production data quality or real-world customer behavior.

### Responsible use

The score should be used as a prioritization aid, not as an automatic decision about a customer. Production use would require privacy controls, fairness checks, monitoring, human review, calibrated probabilities, and measured experimentation.

## 12. Project Deliverables and Roadmap

### Current deliverables

- Exploratory data analysis notebook
- Churn model training and evaluation notebook
- Saved XGBoost preprocessing-and-model pipeline
- Streamlit app for single-customer prediction
- Project documentation and local run instructions

### Recommended next steps

1. Complete behavioral customer segmentation using clustering and profile the resulting groups.
2. Add explainability so users can understand the main factors associated with a prediction.
3. Add batch scoring and export for a retention operations workflow.
4. Calibrate probabilities and select a threshold using campaign economics.
5. Validate fairness and performance across customer groups.
6. Run controlled retention experiments to measure incremental business impact.
7. Add monitoring for data drift, model performance, and retraining needs.

## Conclusion

The core business opportunity is to move from reactive churn reporting to proactive, evidence-based retention prioritization. This project addresses that opportunity by combining customer behavior data, exploratory analysis, supervised classification, and an accessible Streamlit interface.

The result is a practical prototype: it demonstrates how a business question can become a measurable analytics workflow and a usable decision-support application. The next level of maturity is not simply a more complex model; it is connecting predictions to ethical, targeted interventions and verifying that those interventions create incremental customer and business value.